In [21]:
import pandas as pd

df = pd.read_csv("sentimentDataset\sentiment_analysis.csv")
df = df[['text', 'sentiment']]   # keep only what we need

print(df.head())
print(df['sentiment'].value_counts())


                                                text sentiment
0              What a great day!!! Looks like dream.  positive
1     I feel sorry, I miss you here in the sea beach  positive
2                                     Don't angry me  negative
3  We attend in the class just for listening teac...  negative
4                  Those who want to go, let them go  negative
sentiment
neutral     199
positive    166
negative    134
Name: count, dtype: int64


In [22]:
#Text Cleaning

import re

def clean_text(text):
    text = text.lower()                      # lowercase
    text = re.sub(r'http\S+', '', text)      # remove links
    text = re.sub(r'[^a-z\s]', '', text)     # remove punctuation & numbers
    text = re.sub(r'\s+', ' ', text).strip() # remove extra spaces
    return text

df['clean_text'] = df['text'].apply(clean_text)
df.head()


,text,sentiment,clean_text
0,What a great day!!! Looks like dream.,positive,what a great day looks like dream
1,"I feel sorry, I miss you here in the sea beach",positive,i feel sorry i miss you here in the sea beach
2,Don't angry me,negative,dont angry me
3,We attend in the class just for listening teac...,negative,we attend in the class just for listening teac...
4,"Those who want to go, let them go",negative,those who want to go let them go


In [30]:
#label Encoding
df['sentiment'] = df['sentiment'].str.lower().str.strip()

df['label'] = df['sentiment'].map({
    'positive': 2,
    'neutral': 1,
    'negative': 0
})
df.head()


,text,sentiment,clean_text,label
0,What a great day!!! Looks like dream.,positive,what a great day looks like dream,2
1,"I feel sorry, I miss you here in the sea beach",positive,i feel sorry i miss you here in the sea beach,2
2,Don't angry me,negative,dont angry me,0
3,We attend in the class just for listening teac...,negative,we attend in the class just for listening teac...,0
4,"Those who want to go, let them go",negative,those who want to go let them go,0


In [ ]:
df = df.dropna(subset=['label']) # drop rows where label is NaN

print(df['label'].value_counts())
print(df['label'].isnull().sum())


label
1    199
2    166
0    134
Name: count, dtype: int64
0


In [32]:
#Tokenization and Vectorization
#convert text to numerical format using TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,   # limit vocabulary
    ngram_range=(1,2)    # words + word pairs
)

X = vectorizer.fit_transform(df['clean_text'])
y = df['label']

print(X.shape) #displayes number of samples and features
print(y.shape) #displayes number of labels


(499, 4623)
(499,)


In [33]:
#Split Data

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, #80% for training, 20% for testing
    random_state=42,
    stratify=y
)


In [35]:
#training the model

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    multi_class='auto'
)

model.fit(X_train, y_train)


c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'auto'


In [38]:
#testing the model
y_pred = model.predict(X_test)

#evaluate the model 
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=['Negative', 'Neutral', 'Positive']
))



Accuracy: 0.63
              precision    recall  f1-score   support

    Negative       1.00      0.19      0.31        27
     Neutral       0.52      0.93      0.67        40
    Positive       0.88      0.64      0.74        33

    accuracy                           0.63       100
   macro avg       0.80      0.58      0.57       100
weighted avg       0.77      0.63      0.59       100



In [ ]:
#comparing with Naive Bayes
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))
print(classification_report(
    y_test,
    nb_pred,
    target_names=['Negative', 'Neutral', 'Positive']
))


Naive Bayes Accuracy: 0.64
              precision    recall  f1-score   support

    Negative       1.00      0.19      0.31        27
     Neutral       0.53      0.93      0.67        40
    Positive       0.88      0.67      0.76        33

    accuracy                           0.64       100
   macro avg       0.80      0.59      0.58       100
weighted avg       0.77      0.64      0.60       100



Both Logistic Regression and Multinomial Naive Bayes achieved similar performance on the multiclass sentiment task. Naive Bayes marginally outperformed Logistic Regression in overall accuracy and weighted F1-score, indicating its suitability for probabilistic text classification with TF-IDF features.

In [42]:
#use LSTM

label_map = {'negative': 0, 'neutral': 1, 'positive': 2}

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

#create tokenizer
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_text'])

#convert text to sequences
sequences = tokenizer.texts_to_sequences(df['clean_text'])



In [43]:
#Padding sequences

max_len = 100

X_seq = pad_sequences(
    sequences,
    maxlen=max_len,
    padding='post',
    truncating='post'
)


In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_seq,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [58]:
#Build LSTM model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=max_len),
    LSTM(32, return_sequences=False),
    Dropout(0.5),
    Dense(3, activation='softmax')
])


In [59]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


In [60]:
history = model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2
)


Epoch 1/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 110ms/step - accuracy: 0.4013 - loss: 1.0915 - val_accuracy: 0.3625 - val_loss: 1.1059
Epoch 2/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.4013 - loss: 1.0806 - val_accuracy: 0.3625 - val_loss: 1.1348
Epoch 3/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.3824 - loss: 1.0874 - val_accuracy: 0.3625 - val_loss: 1.1251
Epoch 4/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 55ms/step - accuracy: 0.3730 - loss: 1.0868 - val_accuracy: 0.3625 - val_loss: 1.1191
Epoch 5/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.3981 - loss: 1.0792 - val_accuracy: 0.3625 - val_loss: 1.1244
Epoch 6/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 60ms/step - accuracy: 0.3981 - loss: 1.0788 - val_accuracy: 0.3625 - val_loss: 1.1281
Epoch 7/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - accuracy: 0.4075 - loss: 1.0820 - val_accuracy: 0.3625 - val_loss: 1.1226
Epoch 8/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - accuracy: 0.4044 - loss: 1.0822 - val_accuracy: 0.3625 - 

In [61]:
loss, accuracy = model.evaluate(X_test, y_test)
print("LSTM Test Accuracy:", accuracy)


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.4000 - loss: 1.0864
LSTM Test Accuracy: 0.4000000059604645


In [62]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))


In [63]:
model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights
)


Epoch 1/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step - accuracy: 0.4075 - loss: 1.0953 - val_accuracy: 0.3625 - val_loss: 1.1120
Epoch 2/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - accuracy: 0.3824 - loss: 1.0971 - val_accuracy: 0.2750 - val_loss: 1.1070
Epoch 3/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.3668 - loss: 1.0946 - val_accuracy: 0.2750 - val_loss: 1.1051
Epoch 4/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step - accuracy: 0.3887 - loss: 1.0863 - val_accuracy: 0.2750 - val_loss: 1.1058
Epoch 5/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 84ms/step - accuracy: 0.3542 - loss: 1.0917 - val_accuracy: 0.2750 - val_loss: 1.1089
Epoch 6/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - accuracy: 0.4138 - loss: 1.0924 - val_accuracy: 0.3625 - val_loss: 1.1043
Epoch 7/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.3950 - loss: 1.0903 - val_accuracy: 0.3625 - val_loss: 1.1029
Epoch 8/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.3856 - loss: 1.0863 - val_accuracy: 0.3625 - v

In [64]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Improved LSTM Accuracy:", accuracy)


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3300 - loss: 1.0930
Improved LSTM Accuracy: 0.33000001311302185
